# [10.1] Capstone Research Sprint - Exercises

Build a capstone research sprint: plan fields, baseline completeness, causal validations, reproducibility metadata, and a live mini activation-oracle report contract.


In [ ]:
import json
import sys
from dataclasses import dataclass
from pathlib import Path

import torch as t

chapter = "chapter10_capstone_research_sprint"
section = "part1_capstone_research_sprint"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part1_capstone_research_sprint.tests as tests
import part1_capstone_research_sprint.utils as utils

GT_TIER = "GT-4"
EXERCISE_ID = "10_1_capstone_research_sprint"
DIFFICULTY = 5
IMPORTANCE = 3
EXPECTED_RUNTIME = "25-40 minutes for exercises; seconds for the CUDA mini-capstone report"
REQUIRES_GPU = True

In [ ]:
@dataclass(frozen=True)
class CapstonePlan:
    research_question: str
    benchmark: str
    baselines: tuple[str, ...]
    mechanistic_claim: str
    causal_validations: tuple[str, ...]
    reproducible_scripts: tuple[str, ...]
    writeup_path: str


@dataclass(frozen=True)
class BaselineSuiteReport:
    required_baselines: tuple[str, ...]
    present_baselines: tuple[str, ...]
    missing_baselines: tuple[str, ...]
    complete: bool


@dataclass(frozen=True)
class CausalValidationSuiteReport:
    validations: tuple[str, ...]
    has_ablation: bool
    has_patching: bool
    has_random_control: bool
    has_ood: bool
    complete: bool


@dataclass(frozen=True)
class ReproducibilityReport:
    script_paths: tuple[str, ...]
    seeds: tuple[int, ...]
    artifact_paths: tuple[str, ...]
    reproducible: bool


@dataclass(frozen=True)
class CapstoneReadinessReport:
    has_research_question: bool
    has_benchmark: bool
    has_mechanistic_claim: bool
    baseline_suite_complete: bool
    causal_validation_complete: bool
    reproducibility_complete: bool
    has_writeup_path: bool
    ready: bool

## Capstone Plan

Strip blank strings, keep nonblank entries in order, and freeze repeated fields as tuples.

In [ ]:
def build_capstone_plan(
    *,
    research_question: str,
    benchmark: str,
    baselines: list[str],
    mechanistic_claim: str,
    causal_validations: list[str],
    reproducible_scripts: list[str],
    writeup_path: str,
) -> CapstonePlan:
    raise NotImplementedError()


tests.test_build_capstone_plan_normalizes_blank_fields(build_capstone_plan)

In [ ]:
def _example_plan() -> CapstonePlan:
    return build_capstone_plan(
        research_question="Do mini Activation Oracles beat probes?",
        benchmark="held-out activation questions",
        baselines=["probe", "text_only", "random_control"],
        mechanistic_claim="question conditioning uses latent state features",
        causal_validations=["ablation", "patching", "random_control", "ood"],
        reproducible_scripts=["scripts/run_capstone.py"],
        writeup_path="reports/capstone.md",
    )


def plan_smoke_test() -> dict:
    return _example_plan().__dict__

## Baseline Suite

Require probe, text-only, and random-control baselines unless a stricter local contract is passed in.

In [ ]:
def baseline_suite_report(
    present_baselines: list[str],
    *,
    required_baselines: tuple[str, ...] = ("probe", "text_only", "random_control"),
) -> BaselineSuiteReport:
    raise NotImplementedError()


tests.test_baseline_suite_report_identifies_missing_required_baseline(
    baseline_suite_report,
)


def baseline_smoke_test() -> dict:
    plan = _example_plan()
    return baseline_suite_report(list(plan.baselines)).__dict__


tests.test_baseline_smoke_test_has_required_controls(baseline_smoke_test)

## Causal Validation Suite

Require ablation, patching, random control, and OOD or held-out-template validation.

In [ ]:
def causal_validation_suite_report(
    validations: list[str],
) -> CausalValidationSuiteReport:
    raise NotImplementedError()


tests.test_causal_validation_suite_report_accepts_equivalent_names(
    causal_validation_suite_report,
)


def validation_smoke_test() -> dict:
    plan = _example_plan()
    return causal_validation_suite_report(list(plan.causal_validations)).__dict__

## Reproducibility Gate

A rerunnable capstone needs at least one script, one seed, and one declared output artifact.

In [ ]:
def reproducibility_report(
    *,
    script_paths: list[str],
    seeds: list[int],
    artifact_paths: list[str],
    root: str | Path | None = None,
) -> ReproducibilityReport:
    raise NotImplementedError()


tests.test_reproducibility_report_requires_scripts_seeds_and_artifacts(
    reproducibility_report,
)


def reproducibility_smoke_test() -> dict:
    plan = _example_plan()
    return reproducibility_report(
        script_paths=list(plan.reproducible_scripts),
        seeds=[0, 1, 2],
        artifact_paths=["results/metrics.json"],
        root=section_dir,
    ).__dict__


## Readiness Gate

Combine the plan fields, baseline report, validation report, reproducibility report, and writeup path into one readiness decision.

In [ ]:
def capstone_readiness_report(
    plan: CapstonePlan,
    baselines: BaselineSuiteReport,
    validations: CausalValidationSuiteReport,
    reproducibility: ReproducibilityReport,
) -> CapstoneReadinessReport:
    raise NotImplementedError()


tests.test_capstone_readiness_report_requires_every_gate(
    build_capstone_plan,
    baseline_suite_report,
    causal_validation_suite_report,
    reproducibility_report,
    capstone_readiness_report,
)


def readiness_smoke_test() -> dict:
    plan = _example_plan()
    baselines = baseline_suite_report(list(plan.baselines))
    validations = causal_validation_suite_report(list(plan.causal_validations))
    reproducibility = reproducibility_report(
        script_paths=list(plan.reproducible_scripts),
        seeds=[0, 1, 2],
        artifact_paths=["results/metrics.json"],
        root=section_dir,
    )
    return capstone_readiness_report(
        plan,
        baselines,
        validations,
        reproducibility,
    ).__dict__

## Notebook Contract

Expose the section-level smoke contract with the same top-level keys as the verification report.

In [ ]:
def run_smoke_test(cpu: bool = True) -> dict:
    _ = cpu
    return {
        "plan": plan_smoke_test(),
        "baselines": baseline_smoke_test(),
        "validations": validation_smoke_test(),
        "reproducibility": reproducibility_smoke_test(),
        "readiness": readiness_smoke_test(),
    }


tests.test_notebook_contract(run_smoke_test)

## CUDA Verification Report

This committed report is a live CUDA mini-capstone: it trains the activation oracle, runs baselines and controls, and writes the result artifacts. The claim remains scoped to the generated latent-state benchmark.


In [ ]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]

assert report["accepted"]
assert report["gt_tier"] == GT_TIER
assert report["notebook_id"] == EXERCISE_ID
assert gpu["cuda_available"]
assert gpu["live_training_executed"]
assert gpu["seed_count"] == 3
assert gpu["baseline_suite_complete"]
assert gpu["causal_validation_complete"]
assert gpu["reproducible"]
assert gpu["ready"]
assert gpu["oracle_accuracy_mean"] >= 0.9
assert gpu["oracle_beats_text_only"]
assert gpu["oracle_beats_linear_probe_bank"]
assert gpu["compositional_oracle_beats_linear_probe"]
assert gpu["heldout_template_accuracy_mean"] >= 0.9
assert gpu["ablation_drop_mean"] > 0.2
assert gpu["counterfactual_patch_target_accuracy_mean"] >= 0.9
assert gpu["random_patch_change_rate_mean"] <= 0.15
assert gpu["random_activation_control_passed"]
assert gpu["label_shuffle_control_passed"]
assert gpu["preflight_passed"]
assert gpu["peak_vram_gb"] <= 1.0
assert gpu["within_vram_budget"]

utils.print_report(
    "10.1 CUDA mini-capstone report",
    {key: gpu[key] for key in [
        "device",
        "seed_count",
        "oracle_accuracy_mean",
        "text_only_accuracy_mean",
        "linear_probe_bank_accuracy_mean",
        "linear_probe_compositional_accuracy_mean",
        "heldout_template_accuracy_mean",
        "ablation_drop_mean",
        "counterfactual_patch_target_accuracy_mean",
        "random_patch_change_rate_mean",
        "random_activation_accuracy_mean",
        "label_shuffle_accuracy_mean",
        "preflight_passed",
        "peak_vram_gb",
    ]},
)


## Full Verification Contract

The smoke tests check the local exercise implementation. This final cell checks the committed CUDA verification report and exposes a lightweight notebook shim with the same return shape. The release gate itself calls `solutions.run_gpu_test`, which reruns `scripts/run_capstone.py` on CUDA.


In [ ]:
def _load_committed_gpu_report() -> dict:
    import json

    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["live_training_executed"]
    assert gpu["metrics_by_seed_file_valid"]
    assert gpu["preflight_passed"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = _load_committed_gpu_report()
{key: gpu[key] for key in [
    "device",
    "seed_count",
    "oracle_accuracy_mean",
    "linear_probe_compositional_accuracy_mean",
    "ablation_drop_mean",
    "preflight_passed",
    "peak_vram_gb",
] if key in gpu}
